# M4 Practice: Data Wrangling with NTL-LTER Data

This practice notebook provides additional preparation for Assignment 4. You will use the North Temperate Lakes Long-Term Ecological Research (NTL-LTER) lake chemistry and physics dataset to practice selecting, transforming, combining, summarizing, and reshaping data with pandas.

The exercises are intentionally open-ended. Complete each code cell and inspect your results before moving on. Use the techniques from both M4 lesson notebooks.

## Learning goals

By the end of this practice, you should be able to:

- import a raw environmental dataset and inspect its structure;
- filter rows with Boolean masks, `.query()`, and `.isin()`;
- identify and handle missing values;
- select and rename columns;
- create variables from dates and existing measurements;
- combine compatible dataframes with `pd.concat()`;
- create grouped summaries; and
- reshape data between long and wide formats.

## 1. Set up the session and inspect the raw data

In [ ]:
# 1.1 Import packages
from pathlib import Path
import pandas as pd

In [ ]:
# 1.2 Set folder paths (project, raw, processed)
project_fldr = Path.cwd().parent
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'
raw_fldr

In [ ]:
# 1.3. Read the raw data in as a dataframe
NTL = pd.read_csv(
    raw_fldr / 'NTL-LTER_Lake_ChemistryPhysics_Raw.csv',
    dtype={'lakeid': 'category', 'lakename': 'category'},
    parse_dates=['sampledate'],
    date_format='%m/%d/%y'
)
NTL.head()

### Exercise 1

Use `.shape`, `.info()`, and `.describe()` to inspect the dataset. Then answer in a markdown cell:

1. How many rows and columns are present?
2. Which columns contain dates, categories, and measurements?
3. What is one feature of the dataset that may require wrangling before analysis?
4. Why isn't the columns column converted into a categorical column?

In [ ]:
# Ex1.1 Reveal the dimensions of the dataframe
NTL.shape

In [ ]:
# Ex1.2 Reveal column information on the dataframe
NTL.info()

In [ ]:
# Ex1.3 Reveal summary statistics for numeric columns
NTL.describe()

## 2. Filter and select an analysis subset

Create a subset for three lakes: Peter Lake, Paul Lake, and Tuesday Lake. Keep only surface observations (`depth == 0`) collected from 2000 onward. Use `.loc[]` and `.isin()` for the filter.

Name the result `surface_three_lakes`. Report its dimensions and the number of observations contributed by each lake.
>Note: You can remove unused lakename categories with `surface_three_lakes['lakename'].cat.remove_unused_categories()`

In [ ]:
# 2.1 Filter selected lakes into a new dataframe
# Create a list of the lakes to select
selected_lakes = ['Peter Lake', 'Paul Lake', 'Tuesday Lake']

# Code to create surface_three_lakes here
surface_three_lakes = (
    NTL
    .loc[
        (NTL['lakename'].isin(selected_lakes))&
        (NTL['depth'] == 0) &
        (NTL['year4'] >= 2000)
    ]
)


In [ ]:
# 2.2 Report dimensions and counts by lake
surface_three_lakes.shape

In [ ]:
# 2.3 Report the number of records for each lake 
surface_three_lakes['lakename'].cat.remove_unused_categories().value_counts()

### Exercise 2: Missing values

Using `surface_three_lakes`, determine:

1. How many missing values occur in each measurement column?
2. What percentage of irradiance (water) values are missing?
3. Create `complete_surface_three_lakes`, retaining only rows with non-missing temperature and dissolved oxygen.

Report the number of rows removed.

In [ ]:
# Ex2.1 Reveal the total number of missing values in each column
surface_three_lakes.isna().sum()

In [ ]:
# Ex2.2 Report percentage of irradiance (water) values missing 
surface_three_lakes['irradianceWater'].isna().mean() * 100

In [ ]:
# Ex2.3 Extract complete records into new dataframe & report dimensions
complete_surface_three_lakes = surface_three_lakes.loc[
    surface_three_lakes['temperature_C'].notna() &
    surface_three_lakes['dissolvedOxygen'].notna()
]

surface_three_lakes.shape[0] - complete_surface_three_lakes.shape[0]

## 3. Select, rename, and create variables

Starting with `complete_surface_three_lakes`, keep only the lake name, sample date, depth, temperature, and dissolved oxygen columns. Rename them using snake_case names.

Then create:

- `year` and `month` from `sample_date`;
- `season` using the month-to-season dictionary from M4-2; and
- `temperature_F`, converting Celsius to Fahrenheit with `temperature_C * 9 / 5 + 32`.

Name the result `lake_features`.

In [ ]:
#3.1 Create a season dictionary
season_dict = {
    12:'winter',1:'winter',2:'winter',
    3:'spring',4:'spring',5:'spring',
    6:'summer',7:'summer',8:'summer',
    9:'fall',10:'fall',11:'fall'
}

In [ ]:
# 3.2 Code wrangle complete_surface_three_lakes dataframe
lake_features = (
    #Subset columns
    complete_surface_three_lakes[['lakename','sampledate','depth','temperature_C','dissolvedOxygen']]
    #Create year, month, season, temperature_F column
    .assign(
        year = lambda df: df['sampledate'].dt.year,
        month = lambda df: df['sampledate'].dt.month,
        season = lambda df: df['month'].map(season_dict),
        temperature_F = lambda df: df['temperature_C'] * 9/5+32
    )
    #Rename
    .rename(columns={
        'lakename':'lake_name',
        'depth':'depth_m',
        'sampledate':'sample_date',
        'dissolvedOxygen':'dissolved_oxygen_mgL'
    })
)

In [ ]:
# 3.3 Preview the transformed data
lake_features.head()

## 4. Practice `pd.concat()`

`pd.concat()` stacks compatible dataframes. Here, create two dataframes from `lake_features`:

- `early_lakes`: observations from 2000 through 2009;
- `late_lakes`: observations from 2010 onward.

Use `pd.concat()` to recombine them into `recombined_lakes`. Set `ignore_index=True`. Compare the dimensions and lake counts of the original and recombined dataframes.

Explain in a markdown cell why `concat()` is appropriate here and why `merge()` would not be the primary operation.

In [ ]:
# 4.1 Create early_lakes and late_lakes dataframes
early_lakes = lake_features[lake_features['year'] <= 2009]
late_lakes = lake_features[lake_features['year'] >= 2010]
early_lakes.shape, late_lakes.shape

In [ ]:
# 4.2 Recombine the dataframes with pd.concat()
recombined_lakes = pd.concat([early_lakes,late_lakes])
recombined_lakes.shape

## 5. Grouped summaries

Using `recombined_lakes`, create a dataframe named `season_summary` with one row for each lake and season. Include:

- the number of records;
- mean temperature in Celsius;
- mean temperature in Fahrenheit; and
- mean dissolved oxygen.

Reset the index and sort the result by lake name and season.

In [ ]:
# 5.1 Group, summarize, and sort combined lake data
season_summary = (
    recombined_lakes
    .groupby(['lake_name','season'])
    .agg(
        record_count = ('temperature_C','count'),
        mean_temperature_C = ('temperature_C','mean'),
        mean_temperature_F = ('temperature_F','mean'),
        mean_dissolved_oxygen = ('dissolved_oxygen_mgL','mean')
    )
    .reset_index()
    .sort_values(['lake_name','season'])
)

In [ ]:
# 5.2 Examine results
season_summary

## 6. Reshape a summary from long to wide

Use `.pivot_table()` to create `season_temperature_wide`, with one row per lake and one column per season. The values should be mean temperature in Celsius.

Then use `.melt()` to convert it back to a long dataframe named `season_temperature_long`. Keep `lake_name` as an identifier and name the new columns `season` and `mean_temperature`.

In [ ]:
# 6.1 Create season_temperature_wide
season_temperature_wide = (
    season_summary
    .pivot_table(
        index='lake_name',
        columns='season',
        values='mean_temperature_C'
    )
    .reset_index()
)

In [ ]:
# 6.2 Create season_temperature_long
season_temperature_long = (
    season_temperature_wide
    .melt(
        id_vars = 'lake_name',
        value_vars = ['spring','summer','fall'],
        value_name = 'Mean Temperture C',
        var_name='Season'
    )
)


## 7. Capstone practice workflow

Create `practice_summary` in one readable workflow. It should:

1. start with `NTL`;
1. filter to Peter Lake, Paul Lake, and Tuesday Lake;
1. keep surface observations with non-missing temperature and dissolved oxygen;
1. create `year`, `month`, and `season`;
1. group by lake and year;
1. calculate record count, mean temperature, and mean dissolved oxygen;
1. reset the index; and
1. sort by lake and year.

Finally, save it as `NTL_practice_summary.csv` in the processed-data folder.

In [ ]:
# Build practice_summary here
NTL_output = (
    NTL
    .loc[( #Filter for lake names, depth, temperature, and DO
            (NTL['lakename'].isin(['Peter Lake','Paul Lake'])) & 
            (NTL['depth']==0) &
            (NTL['temperature_C'].notna() &
             NTL['dissolvedOxygen'].notna())
        )]
    .assign(
        year = lambda df: df['sampledate'].dt.year,
        month = lambda df: df['sampledate'].dt.month,
        season=lambda df: df['month'].map(season_dict),
    )
    .groupby(['lakename','year'])
    .agg(
        record_count = ('temperature_C','count'),
        mean_temp = ('temperature_C','mean'),
        mean_DO = ('dissolvedOxygen','mean')
    )
    .reset_index()
    .sort_values(['lakename','year'])
)
NTL_output.head()

In [ ]:
# Export the processed summary here
NTL_output.to_csv(processed_fldr/'NTL_practice_summary.csv',index=False)

## Reflection

In a markdown cell, briefly answer:

1. Which step required the most judgment?
2. How did filtering before summarizing affect the result?
3. What would you check before trusting a processed dataset created from several raw files?